# LabOne Q — Extra Lesson: Installing a Multi-Qubit System on a ZQCS

The previous lectures ran on Gen2 instruments (SHFQC, HDAWG, PQSC). This lesson is what the first day
with a **ZQCS**, Zurich Instruments' Gen4 control system, looks like — in the order it happens:

1. **The device setup, line by line** — every logical signal of a 4 × 5 lattice of flux-tunable transmons
   with tunable couplers, written out with the Python device-setup API, one ZQCS port per line
2. **Line calibration** — LOs, IFs and ranges for every line, then *save it once, load it everywhere*
3. **A QPU on top, and the first three experiments** — `resonator_spectroscopy`, `amplitude_rabi` and
   `ramsey` from the Applications Library, on many qubits at once
4. **Every line at once, in the plain DSL** — an installation smoke test that fires every flux line, every
   drive line and every readout of the system, written in full LabOne Q DSL with no helper and no workflow,
   to show what the DSL looks like underneath everything the library builds

**This notebook needs the ZQCS device driver** (`zhinst-zqcs`, shipped with a ZQCS installation — not on
PyPI). On the open LabOne Q release the other lectures use, `session.connect` stops with
`Unknown device driver ZQCS`. Everything else — the setup API, the calibration, the DSL — is the same
LabOne Q. With the driver installed it runs in emulation (`use_emulation = True`) or on the rack.

In [ ]:
import numpy as np

from laboneq.simple import *
from laboneq import serializers
from laboneq.dsl.device.instruments import ZQCS
from laboneq.contrib.example_helpers.plotting.plot_helpers import plot_simulation

## 1. The device setup, line by line

A ZQCS is one instrument reached through its controller service — no data server. A port is
`shelf:slot:board:channel`: an SG or flux front end has channels 1–7; a QA front end has two QA channels,
output 1 or 2 read back on input 3 or 4.

Until the chip mapping is known, the qubits are a placeholder **4 × 5 square lattice**: 20 flux-tunable
qubits and 31 tunable couplers, the largest such lattice 56 flux lines can hold. Qubit `q{i}`
(row `i // 5`, column `i % 5`) takes drive channel `i`, flux channel `i` and QA channel `i % 16`, so
`q0`–`q3` share their readout line with `q16`–`q19`. The couplers take the next flux channels, and the
channels the lattice leaves free are kept as `spare` lines, so the smoke test at the end reaches every
output of the system.

Every logical signal is one `create_connection` line, with the port it is cabled to. This *is* the
cabling map: to rewire, edit the ports here.

In [ ]:
ZQCS_ADDRESS = "http://zqcs-controller.local"  # the controller service of your installation

setup = DeviceSetup(uid="lattice_4x5_installation")
setup.add_instruments(ZQCS(uid="zqcs", address=ZQCS_ADDRESS))

The 20 qubits, each with a flux, drive, measure (QA output) and acquire (QA input) line:

In [ ]:
# fmt: off
setup.add_connections(
    "zqcs",
    # q0: lattice row 0, column 0
    create_connection(to_signal="q0/flux",     ports="1:1:1:1",  type="rf"),
    create_connection(to_signal="q0/drive",    ports="1:10:1:1", type="iq"),
    create_connection(to_signal="q0/measure",  ports="1:10:2:1", type="iq"),
    create_connection(to_signal="q0/acquire",  ports="1:10:2:3", type="acquire"),
    # q1: lattice row 0, column 1
    create_connection(to_signal="q1/flux",     ports="1:1:1:2",  type="rf"),
    create_connection(to_signal="q1/drive",    ports="1:10:1:2", type="iq"),
    create_connection(to_signal="q1/measure",  ports="1:10:2:2", type="iq"),
    create_connection(to_signal="q1/acquire",  ports="1:10:2:4", type="acquire"),
    # q2: lattice row 0, column 2
    create_connection(to_signal="q2/flux",     ports="1:1:1:3",  type="rf"),
    create_connection(to_signal="q2/drive",    ports="1:10:1:3", type="iq"),
    create_connection(to_signal="q2/measure",  ports="1:10:3:1", type="iq"),
    create_connection(to_signal="q2/acquire",  ports="1:10:3:3", type="acquire"),
    # q3: lattice row 0, column 3
    create_connection(to_signal="q3/flux",     ports="1:1:1:4",  type="rf"),
    create_connection(to_signal="q3/drive",    ports="1:10:1:4", type="iq"),
    create_connection(to_signal="q3/measure",  ports="1:10:3:2", type="iq"),
    create_connection(to_signal="q3/acquire",  ports="1:10:3:4", type="acquire"),
    # q4: lattice row 0, column 4
    create_connection(to_signal="q4/flux",     ports="1:1:1:5",  type="rf"),
    create_connection(to_signal="q4/drive",    ports="1:10:1:5", type="iq"),
    create_connection(to_signal="q4/measure",  ports="1:13:2:1", type="iq"),
    create_connection(to_signal="q4/acquire",  ports="1:13:2:3", type="acquire"),
    # q5: lattice row 1, column 0
    create_connection(to_signal="q5/flux",     ports="1:1:1:6",  type="rf"),
    create_connection(to_signal="q5/drive",    ports="1:10:1:6", type="iq"),
    create_connection(to_signal="q5/measure",  ports="1:13:2:2", type="iq"),
    create_connection(to_signal="q5/acquire",  ports="1:13:2:4", type="acquire"),
    # q6: lattice row 1, column 1
    create_connection(to_signal="q6/flux",     ports="1:1:1:7",  type="rf"),
    create_connection(to_signal="q6/drive",    ports="1:10:1:7", type="iq"),
    create_connection(to_signal="q6/measure",  ports="1:13:3:1", type="iq"),
    create_connection(to_signal="q6/acquire",  ports="1:13:3:3", type="acquire"),
    # q7: lattice row 1, column 2
    create_connection(to_signal="q7/flux",     ports="1:1:2:1",  type="rf"),
    create_connection(to_signal="q7/drive",    ports="1:10:4:1", type="iq"),
    create_connection(to_signal="q7/measure",  ports="1:13:3:2", type="iq"),
    create_connection(to_signal="q7/acquire",  ports="1:13:3:4", type="acquire"),
    # q8: lattice row 1, column 3
    create_connection(to_signal="q8/flux",     ports="1:1:2:2",  type="rf"),
    create_connection(to_signal="q8/drive",    ports="1:10:4:2", type="iq"),
    create_connection(to_signal="q8/measure",  ports="2:3:2:1",  type="iq"),
    create_connection(to_signal="q8/acquire",  ports="2:3:2:3",  type="acquire"),
    # q9: lattice row 1, column 4
    create_connection(to_signal="q9/flux",     ports="1:1:2:3",  type="rf"),
    create_connection(to_signal="q9/drive",    ports="1:10:4:3", type="iq"),
    create_connection(to_signal="q9/measure",  ports="2:3:2:2",  type="iq"),
    create_connection(to_signal="q9/acquire",  ports="2:3:2:4",  type="acquire"),
    # q10: lattice row 2, column 0
    create_connection(to_signal="q10/flux",    ports="1:1:2:4",  type="rf"),
    create_connection(to_signal="q10/drive",   ports="1:10:4:4", type="iq"),
    create_connection(to_signal="q10/measure", ports="2:3:3:1",  type="iq"),
    create_connection(to_signal="q10/acquire", ports="2:3:3:3",  type="acquire"),
    # q11: lattice row 2, column 1
    create_connection(to_signal="q11/flux",    ports="1:1:2:5",  type="rf"),
    create_connection(to_signal="q11/drive",   ports="1:10:4:5", type="iq"),
    create_connection(to_signal="q11/measure", ports="2:3:3:2",  type="iq"),
    create_connection(to_signal="q11/acquire", ports="2:3:3:4",  type="acquire"),
    # q12: lattice row 2, column 2
    create_connection(to_signal="q12/flux",    ports="1:1:2:6",  type="rf"),
    create_connection(to_signal="q12/drive",   ports="1:10:4:6", type="iq"),
    create_connection(to_signal="q12/measure", ports="2:9:2:1",  type="iq"),
    create_connection(to_signal="q12/acquire", ports="2:9:2:3",  type="acquire"),
    # q13: lattice row 2, column 3
    create_connection(to_signal="q13/flux",    ports="1:1:2:7",  type="rf"),
    create_connection(to_signal="q13/drive",   ports="1:10:4:7", type="iq"),
    create_connection(to_signal="q13/measure", ports="2:9:2:2",  type="iq"),
    create_connection(to_signal="q13/acquire", ports="2:9:2:4",  type="acquire"),
    # q14: lattice row 2, column 4
    create_connection(to_signal="q14/flux",    ports="1:1:3:1",  type="rf"),
    create_connection(to_signal="q14/drive",   ports="1:13:1:1", type="iq"),
    create_connection(to_signal="q14/measure", ports="2:9:3:1",  type="iq"),
    create_connection(to_signal="q14/acquire", ports="2:9:3:3",  type="acquire"),
    # q15: lattice row 3, column 0
    create_connection(to_signal="q15/flux",    ports="1:1:3:2",  type="rf"),
    create_connection(to_signal="q15/drive",   ports="1:13:1:2", type="iq"),
    create_connection(to_signal="q15/measure", ports="2:9:3:2",  type="iq"),
    create_connection(to_signal="q15/acquire", ports="2:9:3:4",  type="acquire"),
    # q16: lattice row 3, column 1
    create_connection(to_signal="q16/flux",    ports="1:1:3:3",  type="rf"),
    create_connection(to_signal="q16/drive",   ports="1:13:1:3", type="iq"),
    create_connection(to_signal="q16/measure", ports="1:10:2:1", type="iq"),
    create_connection(to_signal="q16/acquire", ports="1:10:2:3", type="acquire"),
    # q17: lattice row 3, column 2
    create_connection(to_signal="q17/flux",    ports="1:1:3:4",  type="rf"),
    create_connection(to_signal="q17/drive",   ports="1:13:1:4", type="iq"),
    create_connection(to_signal="q17/measure", ports="1:10:2:2", type="iq"),
    create_connection(to_signal="q17/acquire", ports="1:10:2:4", type="acquire"),
    # q18: lattice row 3, column 3
    create_connection(to_signal="q18/flux",    ports="1:1:3:5",  type="rf"),
    create_connection(to_signal="q18/drive",   ports="1:13:1:5", type="iq"),
    create_connection(to_signal="q18/measure", ports="1:10:3:1", type="iq"),
    create_connection(to_signal="q18/acquire", ports="1:10:3:3", type="acquire"),
    # q19: lattice row 3, column 4
    create_connection(to_signal="q19/flux",    ports="1:1:3:6",  type="rf"),
    create_connection(to_signal="q19/drive",   ports="1:13:1:6", type="iq"),
    create_connection(to_signal="q19/measure", ports="1:10:3:2", type="iq"),
    create_connection(to_signal="q19/acquire", ports="1:10:3:4", type="acquire"),
)
# fmt: on

The 31 tunable couplers, one flux line each:

In [ ]:
# fmt: off
setup.add_connections(
    "zqcs",
    create_connection(to_signal="c_q0q1/flux",   ports="1:1:3:7", type="rf"),
    create_connection(to_signal="c_q0q5/flux",   ports="1:1:4:1", type="rf"),
    create_connection(to_signal="c_q1q2/flux",   ports="1:1:4:2", type="rf"),
    create_connection(to_signal="c_q1q6/flux",   ports="1:1:4:3", type="rf"),
    create_connection(to_signal="c_q2q3/flux",   ports="1:1:4:4", type="rf"),
    create_connection(to_signal="c_q2q7/flux",   ports="1:1:4:5", type="rf"),
    create_connection(to_signal="c_q3q4/flux",   ports="1:1:4:6", type="rf"),
    create_connection(to_signal="c_q3q8/flux",   ports="1:1:4:7", type="rf"),
    create_connection(to_signal="c_q4q9/flux",   ports="1:2:1:1", type="rf"),
    create_connection(to_signal="c_q5q6/flux",   ports="1:2:1:2", type="rf"),
    create_connection(to_signal="c_q5q10/flux",  ports="1:2:1:3", type="rf"),
    create_connection(to_signal="c_q6q7/flux",   ports="1:2:1:4", type="rf"),
    create_connection(to_signal="c_q6q11/flux",  ports="1:2:1:5", type="rf"),
    create_connection(to_signal="c_q7q8/flux",   ports="1:2:1:6", type="rf"),
    create_connection(to_signal="c_q7q12/flux",  ports="1:2:1:7", type="rf"),
    create_connection(to_signal="c_q8q9/flux",   ports="1:2:2:1", type="rf"),
    create_connection(to_signal="c_q8q13/flux",  ports="1:2:2:2", type="rf"),
    create_connection(to_signal="c_q9q14/flux",  ports="1:2:2:3", type="rf"),
    create_connection(to_signal="c_q10q11/flux", ports="1:2:2:4", type="rf"),
    create_connection(to_signal="c_q10q15/flux", ports="1:2:2:5", type="rf"),
    create_connection(to_signal="c_q11q12/flux", ports="1:2:2:6", type="rf"),
    create_connection(to_signal="c_q11q16/flux", ports="1:2:2:7", type="rf"),
    create_connection(to_signal="c_q12q13/flux", ports="1:2:3:1", type="rf"),
    create_connection(to_signal="c_q12q17/flux", ports="1:2:3:2", type="rf"),
    create_connection(to_signal="c_q13q14/flux", ports="1:2:3:3", type="rf"),
    create_connection(to_signal="c_q13q18/flux", ports="1:2:3:4", type="rf"),
    create_connection(to_signal="c_q14q19/flux", ports="1:2:3:5", type="rf"),
    create_connection(to_signal="c_q15q16/flux", ports="1:2:3:6", type="rf"),
    create_connection(to_signal="c_q16q17/flux", ports="1:2:3:7", type="rf"),
    create_connection(to_signal="c_q17q18/flux", ports="1:2:4:1", type="rf"),
    create_connection(to_signal="c_q18q19/flux", ports="1:2:4:2", type="rf"),
)
# fmt: on

The channels the lattice leaves free, as `spare` lines:

In [ ]:
# fmt: off
setup.add_connections(
    "zqcs",
    # spare drive channels
    create_connection(to_signal="spare/drive_1_13_1_7", ports="1:13:1:7", type="iq"),
    create_connection(to_signal="spare/drive_1_13_4_1", ports="1:13:4:1", type="iq"),
    create_connection(to_signal="spare/drive_1_13_4_2", ports="1:13:4:2", type="iq"),
    create_connection(to_signal="spare/drive_1_13_4_3", ports="1:13:4:3", type="iq"),
    create_connection(to_signal="spare/drive_1_13_4_4", ports="1:13:4:4", type="iq"),
    create_connection(to_signal="spare/drive_1_13_4_5", ports="1:13:4:5", type="iq"),
    create_connection(to_signal="spare/drive_1_13_4_6", ports="1:13:4:6", type="iq"),
    create_connection(to_signal="spare/drive_1_13_4_7", ports="1:13:4:7", type="iq"),
    create_connection(to_signal="spare/drive_2_3_1_1",  ports="2:3:1:1",  type="iq"),
    create_connection(to_signal="spare/drive_2_3_1_2",  ports="2:3:1:2",  type="iq"),
    create_connection(to_signal="spare/drive_2_3_1_3",  ports="2:3:1:3",  type="iq"),
    create_connection(to_signal="spare/drive_2_3_1_4",  ports="2:3:1:4",  type="iq"),
    create_connection(to_signal="spare/drive_2_3_1_5",  ports="2:3:1:5",  type="iq"),
    create_connection(to_signal="spare/drive_2_3_1_6",  ports="2:3:1:6",  type="iq"),
    create_connection(to_signal="spare/drive_2_3_1_7",  ports="2:3:1:7",  type="iq"),
    create_connection(to_signal="spare/drive_2_3_4_1",  ports="2:3:4:1",  type="iq"),
    create_connection(to_signal="spare/drive_2_3_4_2",  ports="2:3:4:2",  type="iq"),
    create_connection(to_signal="spare/drive_2_3_4_3",  ports="2:3:4:3",  type="iq"),
    create_connection(to_signal="spare/drive_2_3_4_4",  ports="2:3:4:4",  type="iq"),
    create_connection(to_signal="spare/drive_2_3_4_5",  ports="2:3:4:5",  type="iq"),
    create_connection(to_signal="spare/drive_2_3_4_6",  ports="2:3:4:6",  type="iq"),
    create_connection(to_signal="spare/drive_2_3_4_7",  ports="2:3:4:7",  type="iq"),
    create_connection(to_signal="spare/drive_2_9_1_1",  ports="2:9:1:1",  type="iq"),
    create_connection(to_signal="spare/drive_2_9_1_2",  ports="2:9:1:2",  type="iq"),
    create_connection(to_signal="spare/drive_2_9_1_3",  ports="2:9:1:3",  type="iq"),
    create_connection(to_signal="spare/drive_2_9_1_4",  ports="2:9:1:4",  type="iq"),
    create_connection(to_signal="spare/drive_2_9_1_5",  ports="2:9:1:5",  type="iq"),
    create_connection(to_signal="spare/drive_2_9_1_6",  ports="2:9:1:6",  type="iq"),
    create_connection(to_signal="spare/drive_2_9_1_7",  ports="2:9:1:7",  type="iq"),
    create_connection(to_signal="spare/drive_2_9_4_1",  ports="2:9:4:1",  type="iq"),
    create_connection(to_signal="spare/drive_2_9_4_2",  ports="2:9:4:2",  type="iq"),
    create_connection(to_signal="spare/drive_2_9_4_3",  ports="2:9:4:3",  type="iq"),
    create_connection(to_signal="spare/drive_2_9_4_4",  ports="2:9:4:4",  type="iq"),
    create_connection(to_signal="spare/drive_2_9_4_5",  ports="2:9:4:5",  type="iq"),
    create_connection(to_signal="spare/drive_2_9_4_6",  ports="2:9:4:6",  type="iq"),
    create_connection(to_signal="spare/drive_2_9_4_7",  ports="2:9:4:7",  type="iq"),
    # spare flux channels
    create_connection(to_signal="spare/flux_1_2_4_3",   ports="1:2:4:3",  type="rf"),
    create_connection(to_signal="spare/flux_1_2_4_4",   ports="1:2:4:4",  type="rf"),
    create_connection(to_signal="spare/flux_1_2_4_5",   ports="1:2:4:5",  type="rf"),
    create_connection(to_signal="spare/flux_1_2_4_6",   ports="1:2:4:6",  type="rf"),
    create_connection(to_signal="spare/flux_1_2_4_7",   ports="1:2:4:7",  type="rf"),
)
# fmt: on

In [ ]:
qubits = sorted((g for g in setup.logical_signal_groups if g.startswith("q")), key=lambda g: int(g[1:]))
couplers = [g for g in setup.logical_signal_groups if g.startswith("c_")]
spares = list(setup.logical_signal_groups["spare"].logical_signals)
print(len(qubits), "qubits,", len(couplers), "couplers,", len(spares), "spare lines")

# which qubits share a QA channel: group by the measure port
qa_channels = {}
for q in qubits:
    port = setup.logical_signal_groups[q].logical_signals["measure"].physical_channel.uid
    qa_channels.setdefault(port, []).append(q)
for port, channel_qubits in qa_channels.items():
    print(f"{port}: {', '.join(channel_qubits)}")

## 2. Line calibration

One setting per line type. The drive lines run on a HARDWARE oscillator at their maximum output range;
the readout lines on a SOFTWARE oscillator, with one LO per QA channel shared by measure (out) and
acquire (in), and the acquire input at its maximum range. The QA outputs and the flux lines have no
attenuator on the ZQCS, so their range stays unset — the compiler refuses one — and the flux lines need no
calibration at all.

The readout IFs are placeholders spread around 100 MHz on each QA channel, 50 MHz apart where two qubits
share a channel, until the resonators are measured.

In [ ]:
DRIVE_LO = 1e9
READOUT_LO = 1e9  # one LO per QA channel, the same on measure (out) and acquire (in)
DRIVE_RANGE_MAX = 14  # dBm
ACQUIRE_RANGE_MAX = 0  # dBm

readout_ifs = {}
for channel_qubits in qa_channels.values():
    for k, q in enumerate(channel_qubits):
        readout_ifs[q] = 100e6 + 50e6 * (k - (len(channel_qubits) - 1) / 2)

calibration = Calibration()
for q in qubits:
    calibration[f"/logical_signal_groups/{q}/drive"] = SignalCalibration(
        local_oscillator=Oscillator(frequency=DRIVE_LO),
        oscillator=Oscillator(frequency=100e6, modulation_type=ModulationType.HARDWARE),
        range=DRIVE_RANGE_MAX,
    )
    readout_oscillator = Oscillator(frequency=readout_ifs[q], modulation_type=ModulationType.SOFTWARE)
    calibration[f"/logical_signal_groups/{q}/measure"] = SignalCalibration(
        local_oscillator=Oscillator(frequency=READOUT_LO),
        oscillator=readout_oscillator,
    )
    calibration[f"/logical_signal_groups/{q}/acquire"] = SignalCalibration(
        local_oscillator=Oscillator(frequency=READOUT_LO),
        oscillator=readout_oscillator,
        range=ACQUIRE_RANGE_MAX,
    )
setup.set_calibration(calibration)

### 2.1 Save it once, load it everywhere

Writing the setup out line by line and calibrating every line is installation work, done **once**. What it
produces is one file — every line, its port and its calibration — and from then on this is all a notebook
needs:

In [ ]:
serializers.save(setup, "lattice_4x5_setup.json")

setup = serializers.load("lattice_4x5_setup.json")

## 3. Session

In [ ]:
use_emulation = True  # False to run on the ZQCS

session = Session(device_setup=setup)
session.connect(do_emulation=use_emulation)

## 4. The QPU, and the first three experiments

`TunableTransmonQubit.from_device_setup` builds one qubit per logical signal group that follows the
`drive` / `flux` / `measure` / `acquire` convention — the couplers and the spares are left alone. The
parameters are the installation's first guesses; every workflow below reads its pulses and frequencies
from them, and in a real bring-up the analyses write the measured values back (Lecture 6).

One thing the library's qubit does not know is which generation it runs on. Every experiment built with
`@dsl.qubit_experiment` applies `qubit.calibration()` as its experiment calibration, and for a Gen2 rack
that puts the acquisition delay on the acquire line as `port_delay` and a `range` on every line. On the
ZQCS `port_delay` is discontinued — its successor is `delay_signal` — and the QA outputs and the flux lines
take no range. So the qubit for a Gen4 system is a three-line subclass: same parameters, same operations,
a calibration the hardware accepts.

In [ ]:
from laboneq.dsl.quantum import QPU
from laboneq_applications.qpu_types.tunable_transmon import TunableTransmonQubit, TunableTransmonOperations
from laboneq_applications.experiments import resonator_spectroscopy, amplitude_rabi, ramsey

class Gen4TransmonQubit(TunableTransmonQubit):
    def calibration(self):
        calibration = super().calibration()
        acquire = calibration[self.signals["acquire"]]
        acquire.delay_signal, acquire.port_delay = acquire.port_delay, None
        calibration[self.signals["measure"]].range = None
        calibration[self.signals["flux"]].range = None
        calibration[self.signals["measure"]].oscillator.modulation_type = ModulationType.SOFTWARE
        return calibration


transmons = Gen4TransmonQubit.from_device_setup(setup, qubit_uids=qubits)
for i, q in enumerate(transmons):
    q.update(
        drive_lo_frequency=DRIVE_LO,
        resonance_frequency_ge=DRIVE_LO + 100e6 + 10e6 * i,
        readout_lo_frequency=READOUT_LO,
        readout_resonator_frequency=READOUT_LO + readout_ifs[q.uid],
        ge_drive_amplitude_pi=0.5,
        ge_drive_amplitude_pi2=0.25,
        ge_drive_length=32e-9,
        readout_length=1e-6,
        readout_integration_length=1e-6,
        readout_amplitude=0.1,
        drive_range=DRIVE_RANGE_MAX,
        readout_range_in=ACQUIRE_RANGE_MAX,
    )
qpu = QPU(quantum_elements=transmons, quantum_operations=TunableTransmonOperations())
print(transmons[0].signals)

Resonator spectroscopy on the first qubit of each QA channel, then an amplitude Rabi and a Ramsey on all
twenty qubits in parallel — the same `experiment_workflow` calls as Lecture 4, with lists of sweep points,
one per qubit. Analysis is off: in emulation there is no physics to fit, and on the rack the first
spectroscopy is for looking, not fitting.

In [ ]:
first_on_each_channel = [channel_qubits[0] for channel_qubits in qa_channels.values()]

opts = resonator_spectroscopy.experiment_workflow.options()
opts.count(4)
opts.do_analysis(False)

res_spec = resonator_spectroscopy.experiment_workflow(
    session, qpu, first_on_each_channel,
    frequencies=[np.linspace(READOUT_LO - 300e6, READOUT_LO + 300e6, 101)] * len(first_on_each_channel),
    options=opts,
).run()
print([t.name for t in res_spec.tasks])

In [ ]:
opts = amplitude_rabi.experiment_workflow.options()
opts.count(4)
opts.do_analysis(False)

res_rabi = amplitude_rabi.experiment_workflow(
    session, qpu, qubits,
    amplitudes=[np.linspace(0, 1, 21)] * len(qubits),
    options=opts,
).run()

compiled_rabi = res_rabi.tasks["compile_experiment"].output
plot_simulation(compiled_rabi, length=2e-6, signal_names_to_show=["q0/drive", "q19/drive", "q0/measure", "q16/measure"])

In [ ]:
opts = ramsey.experiment_workflow.options()
opts.count(4)
opts.do_analysis(False)

res_ramsey = ramsey.experiment_workflow(
    session, qpu, qubits,
    delays=[np.linspace(0, 2e-6, 21)] * len(qubits),
    detunings=[1e6] * len(qubits),
    options=opts,
).run()

show_pulse_sheet("ramsey_all_qubits", res_ramsey.tasks["compile_experiment"].output)

## 5. Every line at once — the plain DSL

An installation smoke test: flux on every flux line (qubits, couplers, spares), then drive on every drive
line (qubits, spares), then the readout of every qubit — every channel of the system plays in one shot.
No qubit object, no quantum operation, no workflow: one `ExperimentSignal` per line, one `play` per line,
and the sections that order them. Where two qubits share a QA channel their readout tones play together on
the one output and are told apart by their IFs — that is what the multiplexed calibration above is for.

In [ ]:
flux_lines = [f"{g}/flux" for g in qubits + couplers] + [f"spare/{l}" for l in spares if l.startswith("flux")]
drive_lines = [f"{q}/drive" for q in qubits] + [f"spare/{l}" for l in spares if l.startswith("drive")]
print(len(flux_lines), "flux lines,", len(drive_lines), "drive lines,", len(qubits), "readouts")

flux_pulse = pulse_library.const(uid="flux_pulse", length=500e-9, amplitude=0.1)
drive_pulse = pulse_library.gaussian(uid="drive_pulse", length=200e-9, amplitude=0.9)
readout_pulse = pulse_library.const(uid="readout_pulse", length=1e-6, amplitude=0.1)
integration_kernel = pulse_library.const(uid="integration_kernel", length=1e-6, amplitude=1.0)

all_lines = Experiment(
    uid="all_lines",
    signals=[ExperimentSignal(line, map_to=f"/logical_signal_groups/{line}") for line in flux_lines + drive_lines]
    + [ExperimentSignal(f"{q}/measure", map_to=f"/logical_signal_groups/{q}/measure") for q in qubits]
    + [ExperimentSignal(f"{q}/acquire", map_to=f"/logical_signal_groups/{q}/acquire") for q in qubits],
)

with all_lines.acquire_loop_rt(uid="shots", count=100, averaging_mode=AveragingMode.CYCLIC, acquisition_type=AcquisitionType.INTEGRATION):
    with all_lines.section(uid="flux"):
        for line in flux_lines:
            all_lines.play(signal=line, pulse=flux_pulse)
    with all_lines.section(uid="drive", play_after="flux"):
        for line in drive_lines:
            all_lines.play(signal=line, pulse=drive_pulse)
    with all_lines.section(uid="readout", play_after="drive"):
        for q in qubits:
            all_lines.play(signal=f"{q}/measure", pulse=readout_pulse)
            all_lines.acquire(signal=f"{q}/acquire", handle=f"{q}/result", kernel=integration_kernel)
    with all_lines.section(uid="relax", play_after="readout"):
        for q in qubits:
            all_lines.delay(signal=f"{q}/measure", time=100e-6)

In [ ]:
compiled_all_lines = session.compile(all_lines)
results = session.run(compiled_all_lines)

print(len(all_lines.signals), "signals,", f"{compiled_all_lines.estimated_runtime * 1e3:.1f} ms per run")
print(sorted(results.acquired_results.keys()))

A few lines of each kind, side by side: a qubit flux, a coupler flux, a spare, two drives and one
multiplexed readout line with its two tones.

In [ ]:
plot_simulation(
    compiled_all_lines,
    length=3e-6,
    signal_names_to_show=["q0/flux", "c_q0q1/flux", "spare/flux_1_2_4_7", "q0/drive", "q19/drive", "q0/measure", "q16/measure"],
)

And the whole system in the pulse sheet:

In [ ]:
show_pulse_sheet("all_lines", compiled_all_lines)